In [4]:
###Iniciando o spark

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [9]:
spark

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [10]:
from datetime import datetime

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [12]:
dt_proc = datetime.now().strftime('%Y%m%d%H%M%S')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [13]:
def log():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S') + ' >>>'

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [14]:
# Início do processamento da camada trusted de pagamentos


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [15]:
df_pagamentos = spark.read.option('header','true').csv('s3://data-lake-elt-303004020922-us-east-1-an/01_raw/tb_pagamentos/')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [18]:
# Para que serve:  
#Depois disso, você pode executar consultas SQL diretamente sobre esse DataFrame usando spark.sql()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [19]:
df_pagamentos.createOrReplaceTempView("df_pagamentos")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [20]:
qtd_registros = df_pagamentos.count()
print(log(),f'Quantidade de Registros: {qtd_registros}')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

2026-10-10 19:27:35 >>> Quantidade de Registros: 7709

In [24]:
df_pagamentos.printSchema()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- id_pagamento: string (nullable = true)
 |-- id_fatura: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- dt_pagamento: string (nullable = true)
 |-- valor_pagamento: string (nullable = true)

In [23]:
#Definindo o schema para transformar os tipos e salvar a camada trusted

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [28]:
df_pagamentos_formated=spark.sql(f"""
          select
            cast(id_cliente as bigint) as id_cliente,
            cast(id_pagamento as bigint) as id_pagamento,
            cast(id_fatura as bigint) as id_fatura,
            "{dt_proc}" as dt_proc,
            substring(replace(dt_pagamento,'-',''),1,6) as ref,
            to_date(dt_pagamento) as dt_pagamento,
            cast(valor_pagamento as decimal(14,2)) as valor_pagamento
          from df_pagamentos           
          """)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
+------------+---------+----------+------------+---------------+
|id_pagamento|id_fatura|id_cliente|dt_pagamento|valor_pagamento|
+------------+---------+----------+------------+---------------+
|           1|        2|         1|  2023-02-11|        3784.01|
|           2|        3|         1|  2023-03-05|        3265.83|
|           3|        5|         1|  2023-05-15|        2804.65|
|           4|        6|         1|  2023-06-20|        2525.89|
+------------+---------+----------+------------+---------------+

In [29]:
df_pagamentos_formated.createOrReplaceTempView('df_pagamentos_formated')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [30]:
df_pagamentos_formated.printSchema()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- id_cliente: long (nullable = true)
 |-- id_pagamento: long (nullable = true)
 |-- id_fatura: long (nullable = true)
 |-- dt_proc: string (nullable = false)
 |-- ref: string (nullable = true)
 |-- dt_pagamento: date (nullable = true)
 |-- valor_pagamento: decimal(14,2) (nullable = true)

In [ ]:
# O DataFrame já tratado (df_pagamentos_formated) e grava os dados em formato Parquet dentro do bucket S3, 
#   adicionando novos registros sem apagar os antigos, e organizando-os em partições por ref.

#Benefícios dessa abordagem:

# O formato Parquet é otimizado para análise (Spark, Athena, Redshift Spectrum).
# O particionamento por ref acelera consultas que filtram por essa coluna.
# O modo append permite cargas incrementais sem perder histórico.

In [43]:
df_pagamentos_formated.write.mode('append').partitionBy('ref').parquet('s3://data-lake-elt-303004020922-us-east-1-an/02_trusted/tb_0002_pagamentos')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [37]:
# Adicionando uma camada de controle de processamento

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [44]:
df_controle_pagamentos = spark.sql(f"""
           select 
                   'tb_0002_pagamentos' as tb_name,
                   {dt_proc} as dt_proc,
                   {qtd_registros} as qtd_registros
""")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [45]:
df_controle_pagamentos.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------------+--------------+-------------+
|           tb_name|       dt_proc|qtd_registros|
+------------------+--------------+-------------+
|tb_0002_pagamentos|20261010192426|         7709|
+------------------+--------------+-------------+

In [47]:
df_controle_pagamentos.write.mode("append").parquet('s3://data-lake-elt-303004020922-us-east-1-an/04_controle/trunsted')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…